# Phase B: build ONE compressed database and measure retrieval

Neural Nexus, vector-DB compression project. One teammate runs this once per compression scheme.

The only thing you change is `SCHEME` in Cell 2:
`fp32`, `int8`, `int4`, `pq128x8`, `binary`, or `svd256`.

What it does, for every victim model and text length in the Phase A dataset:
1. Builds the compressed database from the fp32 victim vectors.
2. Runs correctness tests (stored size, decode really inverts encode, search agrees with a manual check).
3. Measures retrieval: does the right passage come back for its question (top-1, top-10, top-100)?
4. Saves the stored codes, the decoded vectors (what an attacker gets after stealing the index), and the compressor itself.

How to run:
1. Right panel: **Add Input** -> **Notebooks** -> search `phaseA_fp32_embeddings` (owner Aman Priyansh) -> click +. Add ONLY this one input.
2. Settings: Accelerator = **None** (CPU is enough, saves GPU hours), Internet = **On**.
3. Set `SCHEME` in Cell 2. Rename the notebook to `phaseB-<scheme>` (example `phaseB-int8`).
4. Save Version -> Save & Run All (Commit). About 10-30 minutes (pq128x8 can take up to about 1 hour: training the codebooks is slow on CPU).
5. When done: Share the notebook with Aman (Share -> add username `amanpriyansh`, Can view). Also download the `results_*.json` file from the Output tab and send it on the group.

In [ ]:
# Cell 1 - install
!pip install -q faiss-cpu

In [ ]:
# Cell 2 - settings. CHANGE ONLY SCHEME.
SCHEME = "int8"   # one of: fp32, int8, int4, pq128x8, binary, svd256

import os, glob, json, time, hashlib, datetime
import numpy as np
OUT = "/kaggle/working"
N_QUERY_EVAL = 10_000   # questions used for retrieval scoring
SEED = 0

found = glob.glob("/kaggle/input/**/manifest.json", recursive=True)
assert len(found) == 1, f"Add the phaseA_fp32_embeddings notebook output as the only input (found {found})"
IN = os.path.dirname(found[0])
manifest = json.load(open(found[0]))
assert manifest["phase"] == "A_fp32_embeddings", "wrong input dataset"
assert all(manifest["checks"][k] > 0.2 for k in manifest["checks"] if k.startswith("top10_fp32")), "Phase A checks failed"
print("input:", IN)
print("Phase A positive control BLEU:", manifest["checks"]["positive_control_bleu_L32"])

In [ ]:
# ---- Compression schemes (codecs) ----
# Each codec has three jobs:
#   fit(X)      learn what it needs from the victim database (e.g. PQ codebooks, SVD basis)
#   encode(X)   turn float vectors into what is actually stored (the compressed codes)
#   decode(C)   turn stored codes back into float vectors -- this is what an attacker
#               gets after stealing the index (the codebooks are stored inside the index)
#   search(Q,k) how a real vector DB with this compression answers a query
import numpy as np
import faiss

class Codec:
    name = "base"
    def fit(self, X): return self
    def encode(self, X): raise NotImplementedError
    def decode(self, C): raise NotImplementedError
    def bytes_per_vector(self): raise NotImplementedError
    def build(self, X):
        """Fit on the victim DB, store its compressed codes."""
        self.fit(X)
        self.codes = self.encode(X)
        return self
    def search(self, Q, k):
        raise NotImplementedError


class FP32(Codec):
    """Control: no compression."""
    name = "fp32"
    def fit(self, X):
        self.d = X.shape[1]; return self
    def encode(self, X): return np.ascontiguousarray(X, dtype=np.float32)
    def decode(self, C): return np.asarray(C, dtype=np.float32)
    def bytes_per_vector(self): return 4 * self.d
    def search(self, Q, k):
        index = faiss.IndexFlatIP(self.d)
        index.add(self.codes)
        return index.search(np.ascontiguousarray(Q, dtype=np.float32), k)[1]


class FaissCodec(Codec):
    """int8 / int4 scalar quantisation and product quantisation, all through FAISS.
    Encoding and decoding use the SAME trained index (sa_encode / sa_decode),
    so decode always matches how the DB stored the vector."""
    def __init__(self, name, factory):
        self.name, self.factory = name, factory
    def fit(self, X):
        self.d = X.shape[1]
        self.index = faiss.index_factory(self.d, self.factory, faiss.METRIC_INNER_PRODUCT)
        self.index.train(np.ascontiguousarray(X, dtype=np.float32))
        return self
    def build(self, X):
        self.fit(X)
        self.index.add(np.ascontiguousarray(X, dtype=np.float32))
        self.codes = self.encode(X)
        return self
    def encode(self, X): return self.index.sa_encode(np.ascontiguousarray(X, dtype=np.float32))
    def decode(self, C): return self.index.sa_decode(C)
    def bytes_per_vector(self): return self.index.sa_code_size()
    def search(self, Q, k):
        # Asymmetric search: the query stays float, the DB is compressed.
        # This is what a real FAISS SQ/PQ index does.
        return self.index.search(np.ascontiguousarray(Q, dtype=np.float32), k)[1]


class Binary(Codec):
    """1 bit per dimension: keep only the sign."""
    name = "binary"
    def fit(self, X):
        self.d = X.shape[1]; return self
    def encode(self, X): return np.packbits((X > 0).astype(np.uint8), axis=1)
    def decode(self, C):
        bits = np.unpackbits(C, axis=1)[:, :self.d].astype(np.float32)
        return (2.0 * bits - 1.0) / np.sqrt(self.d)   # +-1, scaled to unit length
    def bytes_per_vector(self): return self.d // 8
    def search(self, Q, k):
        # Hamming search: queries are binarised the same way as the DB.
        index = faiss.IndexBinaryFlat(self.d)
        index.add(self.codes)
        return index.search(self.encode(Q), k)[1]


class SVD(Codec):
    """Keep only the top-r directions of the victim DB (dimensionality reduction)."""
    def __init__(self, r=256):
        self.r = r; self.name = f"svd{r}"
    def fit(self, X):
        self.d = X.shape[1]
        _, _, Vt = np.linalg.svd(X.astype(np.float64), full_matrices=False)
        self.V = Vt[:self.r].T.astype(np.float32)     # d x r
        return self
    def encode(self, X): return np.ascontiguousarray(X @ self.V, dtype=np.float32)
    def decode(self, C): return np.asarray(C, dtype=np.float32)   # attacker sees the r-dim vectors as they are
    def bytes_per_vector(self): return 4 * self.r
    def search(self, Q, k):
        # Queries must go through the same projection, or the dimensions do not match.
        index = faiss.IndexFlatIP(self.r)
        index.add(self.codes)
        return index.search(self.encode(Q), k)[1]


def make_codecs():
    return [
        FP32(),
        FaissCodec("int8", "SQ8"),
        FaissCodec("int4", "SQ4"),
        FaissCodec("pq128x8", "PQ128x8"),   # 128 sub-vectors of 8 dims, 8 bits each
        Binary(),
        SVD(256),
    ]


def roundtrip_check(codec, X):
    """Correctness tests for one codec. Raises if something is wrong."""
    C = codec.encode(X)
    Xr = codec.decode(C)
    out = {"scheme": codec.name, "bytes_per_vector": int(codec.bytes_per_vector())}
    if Xr.shape[1] == X.shape[1]:
        out["rel_error"] = float(np.linalg.norm(X - Xr) / np.linalg.norm(X))
        out["mean_cos_to_original"] = float(np.mean(
            np.sum(X * Xr, 1) / (np.linalg.norm(X, axis=1) * np.linalg.norm(Xr, axis=1) + 1e-12)))
    else:
        out["rel_error"] = float("nan"); out["mean_cos_to_original"] = float("nan")
    # 1) stored code size is what we claim
    assert C.nbytes == len(X) * codec.bytes_per_vector(), (codec.name, C.nbytes)
    # 2) re-encoding a decoded vector gives the same code (decode really inverts encode)
    if codec.name not in ("fp32", "svd256"):
        C2 = codec.encode(Xr)
        out["reencode_match"] = float(np.mean(np.all(C2 == C, axis=1)))
        assert out["reencode_match"] > 0.99, (codec.name, out["reencode_match"])
    # 3) the index's own search agrees with brute force on decoded vectors
    q = X[:200]
    top_index = codec.search(q, 1)[:, 0]
    if codec.name == "binary":
        top_brute = np.argmax(codec.decode(codec.encode(q)) @ codec.decode(codec.codes).T, axis=1)
    elif codec.name.startswith("svd"):
        top_brute = np.argmax(codec.encode(q) @ codec.decode(codec.codes).T, axis=1)
    else:
        top_brute = np.argmax(q @ codec.decode(codec.codes).T, axis=1)
    out["search_agrees"] = float(np.mean(top_index == top_brute))
    assert out["search_agrees"] > 0.95, (codec.name, out["search_agrees"])
    return out


def topk_recall(codec, Q, gold, k=10):
    I = codec.search(Q, k)
    return float(np.mean([gold[i] in I[i] for i in range(len(Q))]))

In [ ]:
# Cell 4 - which (victim model, length) combinations exist in the Phase A dataset
combos = []
for f in sorted(manifest["files"]):
    if f.startswith("bge") and "_victim_L" in f:
        model_name, rest = f[:-4].split("_victim_L")
        combos.append((model_name, int(rest)))
print(combos)

def make_codec(name):
    return {c.name: c for c in make_codecs()}[name]
assert SCHEME in {c.name for c in make_codecs()}, f"unknown SCHEME {SCHEME}"

In [ ]:
# Cell 5 - build, test, measure, save. One block per (model, length).
# Expected decoded-vs-original cosine, from theory and our synthetic tests:
EXPECT_COS = {"fp32": (0.9999, 1.0001), "int8": (0.999, 1.0), "int4": (0.97, 1.0),
              "pq128x8": (0.80, 0.99), "binary": (0.60, 0.95), "svd256": (None, None)}
# Expected top-10 retrieval, written BEFORE the run. fp32 values were measured in Phase A
# (5,000 questions); the other rows are expected drops relative to fp32, from how much each
# scheme distorts the vectors.
FP32_TOP10 = {("bge-large", 32): 0.859, ("bge-large", 128): 0.955, ("bge-base", 32): 0.850}
EXPECT_DROP = {"fp32": (0.00, 0.02), "int8": (0.00, 0.01), "int4": (0.00, 0.04),
               "pq128x8": (0.02, 0.15), "binary": (0.03, 0.20), "svd256": (0.01, 0.10)}
print("Expected for", SCHEME)
print("  decoded-vs-original cosine:", EXPECT_COS[SCHEME])
for (m, L), v in FP32_TOP10.items():
    lo, hi = EXPECT_DROP[SCHEME]
    print(f"  top-10 retrieval {m} L{L}: about {v-hi:.3f} to {v-lo:.3f}")
import faiss
rng = np.random.default_rng(SEED)
results = {"scheme": SCHEME, "created_utc": datetime.datetime.utcnow().isoformat(timespec="seconds"),
           "input_manifest_created": manifest["created_utc"], "faiss": faiss.__version__, "runs": []}

for model_name, L in combos:
    t0 = time.time()
    tag = f"{SCHEME}_{model_name}_L{L}"
    X_vic = np.load(f"{IN}/{model_name}_victim_L{L}.npy")
    X_att = np.load(f"{IN}/{model_name}_att_L{L}.npy")
    Q = np.load(f"{IN}/{model_name}_query.npy")

    codec = make_codec(SCHEME).build(X_vic)
    chk = roundtrip_check(codec, X_vic[:5000])

    lo, hi = EXPECT_COS[SCHEME]
    cos = chk["mean_cos_to_original"]
    chk["cos_in_expected_range"] = None if lo is None else bool(lo <= cos <= hi)
    if SCHEME == "svd256":
        Xr = codec.encode(X_vic[:5000]) @ codec.V.T
        chk["energy_kept"] = float((np.linalg.norm(Xr) / np.linalg.norm(X_vic[:5000])) ** 2)

    # retrieval: question i should find passage i
    qi = rng.choice(len(Q), min(N_QUERY_EVAL, len(Q)), replace=False)
    I = codec.search(Q[qi], 100)
    hit = (I == qi[:, None])
    ret = {f"top{k}": float(hit[:, :k].any(1).mean()) for k in (1, 10, 100)}

    # save: stored codes, decoded vectors (victim and attacker), and the compressor
    np.save(f"{OUT}/codes_{tag}.npy", codec.codes)
    np.save(f"{OUT}/decoded_victim_{tag}.npy", codec.decode(codec.codes).astype(np.float32))
    np.save(f"{OUT}/decoded_att_{tag}.npy", codec.decode(codec.encode(X_att)).astype(np.float32))
    if isinstance(codec, FaissCodec):
        faiss.write_index(codec.index, f"{OUT}/index_{tag}.faiss")
    elif SCHEME == "svd256":
        np.save(f"{OUT}/svd_basis_{tag}.npy", codec.V)

    lo_d, hi_d = EXPECT_DROP[SCHEME]
    ref = FP32_TOP10[(model_name, L)]
    chk["top10_in_expected_range"] = bool(ref - hi_d - 0.02 <= ret["top10"] <= ref - lo_d + 0.02)
    run = {"model": model_name, "length": L, **chk, **{f"retrieval_{k}": v for k, v in ret.items()},
           "n_queries": int(len(qi)), "seconds": round(time.time() - t0, 1)}
    results["runs"].append(run)
    print(json.dumps(run, indent=1))

json.dump(results, open(f"{OUT}/results_{SCHEME}.json", "w"), indent=1)

In [ ]:
# Cell 6 - summary table. Send this (or results_<scheme>.json) to the group.
import pandas as pd
df = pd.DataFrame(results["runs"])
cols = ["model", "length", "bytes_per_vector", "mean_cos_to_original", "cos_in_expected_range",
        "retrieval_top1", "retrieval_top10", "top10_in_expected_range", "retrieval_top100", "search_agrees", "seconds"]
print(f"SCHEME = {SCHEME}")
df[[c for c in cols if c in df.columns]].round(4)

## Reading the table

- `cos_in_expected_range` and `top10_in_expected_range` should be True. If either is False, do not fix anything: send the table to Aman. (False is not automatically a bug: it can be a real result. Aman and Claude decide.)
- `retrieval_top10` for `fp32` is the best possible. Every other scheme is compared against it.
- `search_agrees` should be about 1.0. It checks the compressed index against a manual calculation.